# Topic task

Topic labels in the test set are movie / restaurant / book. There is no single dataset
with those three, so I build the training set from three review sources (one per topic):
movie = rotten_tomatoes, restaurant = yelp, book = amazon book reviews.
Then tf-idf + naive bayes, same idea as the sentiment task.

In [1]:
# need the datasets library for this one
# pip install datasets   (run once in the anaconda prompt with tm311 active)

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import pandas as pd

c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### settings

In [2]:
TEST_PATH = "Sentiment-topic-test.tsv"
N_PER_CLASS = 2000   # how many training texts to take from each source

### get training texts (one source per topic)
streaming=True so we only pull N rows instead of downloading the whole thing.

In [3]:
def take_texts(dataset, field, n):
    texts = []
    for row in dataset:
        t = row.get(field)
        if t and isinstance(t, str) and t.strip():
            texts.append(t.strip())
        if len(texts) >= n:
            break
    return texts

# movie
movie_ds = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train", streaming=True)
movie_texts = take_texts(movie_ds, "text", N_PER_CLASS)

# restaurant (yelp is mostly restaurants)
yelp_ds = load_dataset("fancyzhx/yelp_polarity", split="train", streaming=True)
rest_texts = take_texts(yelp_ds, "text", N_PER_CLASS)

# book reviews
try:
    book_ds = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Books",
                           split="full", streaming=True, trust_remote_code=True)
    book_texts = take_texts(book_ds, "text", N_PER_CLASS)
except Exception as e:
    print("book reviews set unavailable, using amazon_polarity (general product reviews) as book proxy:", type(e).__name__)
    book_ds = load_dataset("fancyzhx/amazon_polarity", split="train", streaming=True)
    book_texts = take_texts(book_ds, "content", N_PER_CLASS)

print("movie:", len(movie_texts), "restaurant:", len(rest_texts), "book:", len(book_texts))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


book reviews set unavailable, using amazon_polarity (general product reviews) as book proxy: RuntimeError
movie: 2000 restaurant: 2000 book: 2000


> **note:** Amazon book-reviews dataset didn’t load properly with our current `datasets` version, so We used `amazon_polarity` for the book class instead. It’s not actually book-only reviews, just general Amazon product reviews, so the results for this class are only an approximation.

### put it together

In [4]:
train_texts = movie_texts + rest_texts + book_texts
train_topics = (["movie"] * len(movie_texts) +
                ["restaurant"] * len(rest_texts) +
                ["book"] * len(book_texts))
print("total training texts:", len(train_texts))

total training texts: 6000


### load test set
use the topic column as gold, ignore sentiment here.

In [5]:
test_df = pd.read_csv(TEST_PATH, sep="\t")
for c in test_df.columns:
    if test_df[c].dtype == object:
        test_df[c] = test_df[c].str.replace("\r", "", regex=False).str.strip()

test_texts = test_df["text"].tolist()
gold_topic = test_df["topic"].tolist()
print(test_df["topic"].value_counts())

topic
movie         5
restaurant    3
book          2
Name: count, dtype: int64


### tf-idf + naive bayes

In [6]:
vec = TfidfVectorizer(min_df=2, stop_words="english")
X_train = vec.fit_transform(train_texts)
X_test = vec.transform(test_texts)

clf = MultinomialNB().fit(X_train, train_topics)
pred = clf.predict(X_test)

for text, p, g in zip(test_texts, pred, gold_topic):
    mark = "" if p == g else "   wrong"
    print(f"[{p:10}] (gold {g:10}){mark}  {text[:60]}")

[movie     ] (gold movie     )  It took eight years for Warner Brothers to recover from the 
[restaurant] (gold restaurant)  All the New York University students love this diner in Soho
[restaurant] (gold restaurant)  This Italian place is really trendy but they have forgotten 
[book      ] (gold book      )  In conclusion, my review of this book would be: I like Jane 
[movie     ] (gold movie     )  The story of this movie is focused on Carl Brashear played b
[book      ] (gold movie     )   wrong  Chris O'Donnell stated that while filming for this movie, he
[restaurant] (gold restaurant)  My husband and I moved to Amsterdam 6 years ago and for as l
[book      ] (gold movie     )   wrong  Dame Maggie Smith performed her role excellently, as she doe
[movie     ] (gold movie     )  The new movie by Mr. Kruno was shot in New York, but the sto
[book      ] (gold book      )  I always have loved English novels, but I just couldn't get 


### results

In [7]:
labels = ["movie", "restaurant", "book"]
print(classification_report(gold_topic, pred, labels=labels, zero_division=0))

              precision    recall  f1-score   support

       movie       1.00      0.60      0.75         5
  restaurant       1.00      1.00      1.00         3
        book       0.50      1.00      0.67         2

    accuracy                           0.80        10
   macro avg       0.83      0.87      0.81        10
weighted avg       0.90      0.80      0.81        10

